# 07 A policy check after the gate: rules as data, a solver that names the rule

The gate decides *which route* a message takes. It does not decide whether Meridian may *act* on a refund claim: that is policy (days before departure, fare type, who cancelled),
and policy belongs in code a reviewer can read. Here the refund policy is a short list of **named rules**, and a solver (Z3) does two jobs that a pile of `if` statements does badly:

1. **Check a proposed action.** If the system wants to approve a full refund and a rule forbids it, the solver returns the **names of the rules that clash** (an unsat core).
2. **Check the rule list itself.** Does any case get two different outcomes (a conflict)? Does any case get none (a gap)? The solver returns a concrete example case.

**What this does not do.** It checks the rules someone wrote; it cannot tell whether they match the written policy. The facts (days to departure, fare paid) must come from a booking system
or from a reviewed extraction step, not from this notebook: here they are structured inputs. Z3 returns an unsat core that is not guaranteed to be minimal, so we name "rules involved", not "the one rule".
No model, no network and no API key is used in this notebook.

In [1]:
import json, time, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="pkg_resources")   # z3 4.12 imports it; harmless
from z3 import Solver, Bool, Int, And, Or, Not, Implies, sat, unsat, BoolVal
import meridian as m

RESULTS = Path("data/results")
print("hardware:", m.hardware())

hardware: Intel Xeon Processor (Skylake, IBRS, no TSX), 4 logical cores, Linux 6.8.0-87-generic


## The rules, as data
Facts: `days` (days before departure when the customer asks), `flex` (flexible fare), `airline` (the airline cancelled). Outcome: `full`, `partial` (50%) or `deny`.
Each rule is a name, a condition and the outcome it requires.

In [2]:
days, flex, airline = Int("days"), Bool("flex"), Bool("airline")
OUTCOMES = ["full", "partial", "deny"]
out = {o: Bool(f"out_{o}") for o in OUTCOMES}

POLICY = [
    ("R1 airline cancelled: refund in full",            airline,                                      "full"),
    ("R2 flexible fare, 2+ days out: refund in full",   And(Not(airline), flex, days >= 2),           "full"),
    ("R3 standard fare, 14+ days out: refund in full",  And(Not(airline), Not(flex), days >= 14),     "full"),
    ("R4 standard fare, 2 to 13 days out: half",        And(Not(airline), Not(flex), days >= 2, days < 14), "partial"),
    ("R5 under 2 days out, customer cancelled: none",   And(Not(airline), days < 2),                  "deny"),
]

def outcome_is(o):
    return And(*[out[x] == (x == o) for x in OUTCOMES])

def rule_formula(rule):
    _, cond, o = rule
    return Implies(cond, outcome_is(o))

## Job 1: check a proposed action and name the rules in the way
A case arrives with facts; the system (the encoder-and-LLM cascade, or a person) proposes an outcome. We assert every rule under its name, add the facts and the proposal, and ask Z3.

In [3]:
def check(facts: dict, proposed: str, policy=POLICY):
    s = Solver()
    for name, cond, o in policy:
        s.assert_and_track(rule_formula((name, cond, o)), name)
    s.add(days == facts["days"], flex == facts["flex"], airline == facts["airline"], outcome_is(proposed))
    t = time.perf_counter(); r = s.check(); ms = (time.perf_counter() - t) * 1000
    return ("allowed", [], ms) if r == sat else ("blocked", sorted(str(c) for c in s.unsat_core()), ms)

cases = [
    ("airline cancelled, full refund",             dict(days=1,  flex=False, airline=True),  "full"),
    ("standard fare, 5 days out, full refund",     dict(days=5,  flex=False, airline=False), "full"),
    ("standard fare, 5 days out, half refund",     dict(days=5,  flex=False, airline=False), "partial"),
    ("flexible fare, 5 days out, full refund",     dict(days=5,  flex=True,  airline=False), "full"),
    ("customer cancels the day before, full",      dict(days=1,  flex=True,  airline=False), "full"),
    ("customer cancels the day before, none",      dict(days=1,  flex=True,  airline=False), "deny"),
]
for label, facts, proposed in cases:
    verdict, core, ms = check(facts, proposed)
    print(f"{label:45s} -> {verdict:8s} {ms:5.1f} ms  {core}")

airline cancelled, full refund                -> allowed   16.5 ms  []


standard fare, 5 days out, full refund        -> blocked    0.6 ms  ['R4 standard fare, 2 to 13 days out: half']
standard fare, 5 days out, half refund        -> allowed    0.5 ms  []
flexible fare, 5 days out, full refund        -> allowed    0.4 ms  []
customer cancels the day before, full         -> blocked    0.5 ms  ['R5 under 2 days out, customer cancelled: none']
customer cancels the day before, none         -> allowed    0.5 ms  []


## Job 2: check the rule list itself
Two questions over *all* possible cases, not a sample. A **conflict** is a case where two rules require different outcomes. A **gap** is a case where no rule applies.
Z3 either proves there is none or hands back an example.

In [4]:
def conflicts(policy):
    found = []
    for i in range(len(policy)):
        for j in range(i + 1, len(policy)):
            (ni, ci, oi), (nj, cj, oj) = policy[i], policy[j]
            if oi == oj: continue
            s = Solver(); s.add(ci, cj, days >= 0, days <= 365)
            if s.check() == sat:
                mdl = s.model(); found.append((ni, nj, {str(d): str(mdl[d]) for d in mdl.decls()}))
    return found

def gaps(policy):
    s = Solver(); s.add(days >= 0, days <= 365, *[Not(cond) for _, cond, _ in policy])
    if s.check() == sat:
        mdl = s.model(); return {str(d): str(mdl[d]) for d in mdl.decls()}
    return None

print("current policy  conflicts:", conflicts(POLICY), "| gap:", gaps(POLICY))

# A plausible edit made in a hurry: the flexible-fare rule loses its "2+ days out" condition.
BUGGY = [r if not r[0].startswith("R2") else ("R2 flexible fare: refund in full", And(Not(airline), flex), "full") for r in POLICY]
print("after the edit  conflicts:")
for c in conflicts(BUGGY): print("  ", c)

# And a policy with a hole: the half-refund rule is deleted.
HOLE = [r for r in POLICY if not r[0].startswith("R4")]
print("rule R4 deleted gap example:", gaps(HOLE))

current policy  conflicts: [] | gap: None
after the edit  conflicts:
   ('R2 flexible fare: refund in full', 'R5 under 2 days out, customer cancelled: none', {'days': '0', 'airline': 'False', 'flex': 'True'})
rule R4 deleted gap example: {'days': '10', 'flex': 'False', 'airline': 'False'}


## Cost and agreement with plain code
For four short rules a Python function gives the same answers, so the solver has to earn its place. Here are both on 5,000 random cases: do they agree, and what does each cost per decision?

In [5]:
def plain(facts):
    if facts["airline"]: return "full"
    if facts["days"] < 2: return "deny"
    if facts["flex"] or facts["days"] >= 14: return "full"
    return "partial"

random.seed(m.SEED)
sample = [dict(days=random.randint(0, 60), flex=random.random() < 0.3, airline=random.random() < 0.15) for _ in range(5000)]
t = time.perf_counter(); expected = [plain(f) for f in sample]; plain_ms = (time.perf_counter() - t) * 1000 / len(sample)

agree, solver_ms = 0, []
for f, e in zip(sample, expected):
    ok = check(f, e)
    agree += ok[0] == "allowed"; solver_ms.append(ok[2])
wrong_blocked = sum(check(f, "full" if e != "full" else "deny")[0] == "allowed" for f, e in zip(sample[:500], expected[:500]))
print(f"solver allows the plain function's outcome on {agree} of {len(sample)} cases")
print(f"solver allows a *different* outcome on {wrong_blocked} of 500 cases (should be 0)")
print(f"median {sorted(solver_ms)[len(solver_ms)//2]:.2f} ms per check with the solver, {plain_ms*1000:.1f} microseconds per case in plain Python")

solver allows the plain function's outcome on 5000 of 5000 cases
solver allows a *different* outcome on 0 of 500 cases (should be 0)
median 0.38 ms per check with the solver, 0.4 microseconds per case in plain Python


## Where a solver beats tests: a long policy and one boundary-value conflict
Five rules do not need a solver. Real policies grow: fare classes, loyalty tiers, regions, channels, amount bands. Below, a generated policy of 48 rules partitions the cases by fare class (6), days-before-departure band (4) and flexible or not (2), so
no two of them overlap. Then someone adds rule 49 for one narrow situation: the highest loyalty tier, the web channel, region 7, a claim of exactly the maximum amount, on the day of departure. It overlaps several of the 48 and asks for a different outcome.
Two ways to find it: run random test cases through the rules, or ask the solver. (A tester who knew to try boundary values would find it too; the point is what a sample of random cases costs, and what the solver costs.)

In [6]:
import numpy as np
fare, band, loyalty, channel, region, amount = Int("fare"), Int("band"), Int("loyalty"), Int("channel"), Int("region"), Int("amount")
BANDS = [(0, 1), (2, 6), (7, 13), (14, 365)]
rng = random.Random(m.SEED)
BIG = []
for f in range(6):
    for b, (lo, hi) in enumerate(BANDS):
        for fx in (False, True):
            BIG.append((f"G{f}.{b}.{int(fx)}", And(fare == f, days >= lo, days <= hi, flex == fx), "deny" if (f, b, fx) == (5, 0, False) else rng.choice(OUTCOMES)))
CONFLICT_RULE = ("G49 top tier, web, region 7, maximum claim, day of departure",
                 And(loyalty == 3, channel == 0, region == 7, amount == 2000, days == 0, fare == 5, flex == False), "full")
FACT_BOUNDS = [days >= 0, days <= 365, fare >= 0, fare <= 5, loyalty >= 0, loyalty <= 3, channel >= 0, channel <= 3, region >= 0, region <= 9, amount >= 0, amount <= 2000]
def conflicts_big(policy):
    found = []
    for i in range(len(policy)):
        for j in range(i + 1, len(policy)):
            (ni, ci, oi), (nj, cj, oj) = policy[i], policy[j]
            if oi == oj: continue
            s = Solver(); s.add(ci, cj, *FACT_BOUNDS)
            if s.check() == sat: found.append((ni, nj))
    return found

t = time.perf_counter(); base_conf = conflicts_big(BIG); t_base = time.perf_counter() - t
t = time.perf_counter(); bug_conf = conflicts_big(BIG + [CONFLICT_RULE]); t_bug = time.perf_counter() - t
print(f"48 rules: {len(base_conf)} conflicts, checked in {t_base:.1f} s over {len(BIG)*(len(BIG)-1)//2} rule pairs")
print(f"49 rules: {len(bug_conf)} conflicts, checked in {t_bug:.1f} s; first: {bug_conf[:2]}")

# Random testing: draw cases uniformly over the same ranges and count how many fall in rule 49 and also in some other rule.
N = 2_000_000
g = np.random.default_rng(m.SEED)
d_, f_, l_, c_, r_, a_ = (g.integers(0, 366, N), g.integers(0, 6, N), g.integers(0, 4, N), g.integers(0, 4, N), g.integers(0, 10, N), g.integers(0, 2001, N))
x_ = g.integers(0, 2, N).astype(bool)
hit = (l_ == 3) & (c_ == 0) & (r_ == 7) & (a_ == 2000) & (d_ == 0) & (f_ == 5) & (~x_)
print(f"random testing: {N:,} cases drawn, {int(hit.sum())} landed inside rule 49 (the chance for one case is about {1/(4*4*10*2001*366*6*2):.1e})")

48 rules: 0 conflicts, checked in 16.4 s over 1128 rule pairs
49 rules: 1 conflicts, checked in 17.8 s; first: [('G5.0.0', 'G49 top tier, web, region 7, maximum claim, day of departure')]


random testing: 2,000,000 cases drawn, 0 landed inside rule 49 (the chance for one case is about 7.1e-10)


## Where a solver beats tests: proving a policy change harms nobody
Marketing changes the standard-fare full-refund threshold from 14 to 21 days, and adds a full refund for the top loyalty tier from 7 days out. The intended effect: standard-fare customers 14 to 20 days out, who are not top tier, now get half instead of full. The property to check is
*no customer gets a worse outcome under the new policy, except that intended group*. A test suite checks the cases someone thought of; the solver searches all of them and either returns a customer who is worse off or says there is none.

In [7]:
RANK = {"deny": 0, "partial": 1, "full": 2}
def outcome_fn(rules_in_priority_order):
    """Outcome rank of a case: the first rule that applies wins, so a prohibition listed first overrides the rest."""
    expr = 0
    for _, cond, o in reversed(rules_in_priority_order):
        from z3 import If
        expr = If(cond, RANK[o], expr)
    return expr
top = Bool("top_tier")
V1 = POLICY
V2 = [POLICY[4], POLICY[0], POLICY[1],
      ("R3 standard fare, 21+ days out: refund in full", And(Not(airline), Not(flex), days >= 21), "full"),
      ("R6 top tier, standard fare, 7+ days out: refund in full", And(Not(airline), Not(flex), top, days >= 7), "full"),
      ("R4 standard fare, 2 to 20 days out: half", And(Not(airline), Not(flex), days >= 2, days < 21), "partial")]
V2_BAD = [r if not r[0].startswith("R5") else ("R5 under 3 days out, customer cancelled: none", And(Not(airline), days < 3), "deny") for r in V2]
# Order matters in V2: the prohibition R5 is checked first, and the top-tier rule R6 before R4.
intended = And(Not(airline), Not(flex), Not(top), days >= 14, days < 21)
def worse_off(new):
    s = Solver(); s.add(days >= 0, days <= 365, outcome_fn(new) < outcome_fn(V1), Not(intended))
    r = s.check()
    if r == unsat: return None
    mdl = s.model(); return {str(dd): str(mdl[dd]) for dd in mdl.decls()}
t = time.perf_counter(); w_ok = worse_off(V2); ms_ok = (time.perf_counter() - t) * 1000
t = time.perf_counter(); w_bad = worse_off(V2_BAD); ms_bad = (time.perf_counter() - t) * 1000
print(f"new policy, anyone worse off outside the intended group? {w_ok}   ({ms_ok:.0f} ms)")
print(f"after a typo (R5 threshold 3 instead of 2): {w_bad}   ({ms_bad:.0f} ms)")

new policy, anyone worse off outside the intended group? None   (32 ms)
after a typo (R5 threshold 3 instead of 2): {'flex': 'True', 'days': '2', 'airline': 'False'}   (37 ms)


In [8]:
res = dict(n_cases=len(sample), agree=agree, wrong_allowed=wrong_blocked,
           solver_median_ms=sorted(solver_ms)[len(solver_ms)//2], plain_us=plain_ms * 1000,
           current_conflicts=len(conflicts(POLICY)), edited_conflicts=conflicts(BUGGY), hole_example=gaps(HOLE),
           long_policy=dict(rules=len(BIG), base_conflicts=len(base_conf), bug_conflicts=len(bug_conf), base_s=t_base, bug_s=t_bug, random_cases=N, random_hits=int(hit.sum())),
           regression=dict(clean=w_ok, typo=w_bad, ms_clean=ms_ok, ms_typo=ms_bad),
           hardware=m.hardware(), z3_version=__import__("z3").get_version_string())
(RESULTS / "07_policy_check.json").write_text(json.dumps(res, indent=2, default=str))

1008

## What to take away
- For a fixed list of five rules, write the function: the solver and the function agree on every sampled case, and the solver costs milliseconds per decision where the function costs microseconds.
- A policy with one conflict at a boundary value (rule 49 above) escapes two million random test cases and is found by checking rule pairs with the solver, in the seconds printed above. A tester who tries boundary values finds it too; the solver does not need to be told where to look.
- A policy change can be checked against a property over every possible customer. "Nobody is worse off except the intended group" came back as proved in milliseconds for the intended change, and as a concrete customer for the typo.
- Limits: the check is only as good as the rules and the facts. The rules must be the ones the policy intends, and facts read out of an email (the days to departure) are a new error source to measure. Rule pairs scale with the square of the rule count; a policy of several hundred rules needs a smarter encoding than the pairwise loop here.